# КТ2 — AI-диспетчер критических инцидентов

Сообщения об инцидентах приходят из производства, ИТ, ритейла и логистики, и пишут их все по-разному. Диспетчер читает каждое, решает, насколько все плохо, кому отдать и что делать первым. На это уходит время, и опасный случай легко потерять среди «срочных» мелочей.

**Что делает система.** Language-модель читает сообщение и готовит карточку: критичность, владелец, первый безопасный шаг, цитата-доказательство, что неизвестно и нужен ли человек.
**Цена ошибок.** Самое дорогое — занизить критичность опасного случая или отдать его не тому владельцу: простой, ущерб, взлом. Завысить мелочь тоже плохо: дежурный бросает настоящую аварию ради «срочно» от одного пользователя. И отдельный риск — модель уверенно выдумывает детали, которых в сообщении не было, а диспетчер ей верит.

Задача: сравнить три версии инструкции (A — короткая, B — структура, C — управляемый AI) на 30 обязательных случаях и понять, какую версию можно пускать в работу и в каком режиме.

**Для Colab.** Ollama и Rust в Colab не запускаю, поэтому ноутбук берет готовые результаты. Загрузить в ячейке ниже:
- `КТ2_отчёты_об_инцидентах.csv`, `llm_raw.jsonl`, `rust_journal.csv`, `rust_summary.csv`, `rust_compare.csv`, `rust_sim.csv`, `допущения_времени.csv` (все из папки `data/`);
- `A_короткая.txt`, `B_структура.txt`, `C_управляемый_AI.txt` (из папки `инструкции/`).

In [ ]:
import os
for folder in ['data', 'img', 'инструкции']:
    os.makedirs(folder, exist_ok=True)
try:
    from google.colab import files
    up = files.upload()
    for name, blob in up.items():
        folder = 'инструкции' if name.endswith('.txt') else 'data'
        with open(os.path.join(folder, name), 'wb') as f:
            f.write(blob)
    print('загружено:', ', '.join(up))
except ImportError:
    print('запущено не в Colab, файлы уже на месте')

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import time
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

DATA = Path('data')
incidents = pd.read_csv(DATA / 'КТ2_отчёты_об_инцидентах.csv')
cases = incidents[incidents.mandatory == 'да'].reset_index(drop=True)

print('сообщений всего:', len(incidents), '| обязательных:', len(cases), '| резерв:', (incidents.mandatory == 'резерв').sum())
print('уникальных текстов среди 30:', cases.report_text.nunique())
display(cases.head(6))
display(pd.crosstab(cases.gold_severity, cases.gold_human_review, margins=True, margins_name='всего'))

In [ ]:
URGENT = 'Автор пишет: «срочно»'
cases['urgent'] = cases.report_text.str.contains(URGENT, regex=False)
cases['base_text'] = cases.report_text.str.replace(r'\s*Автор пишет:.*$', '', regex=True)

templates = (cases.groupby('base_text', sort=False)
             .agg(отрасль=('industry', 'first'), случаев=('incident_id', 'size'), со_срочно=('urgent', 'sum'),
                  severity=('gold_severity', 'first'), owner=('gold_owner', 'first'), first_action=('gold_first_action', 'first'))
             .reset_index())
templates.index = [f'T{i}' for i in range(1, len(templates) + 1)]
cases['template'] = cases.base_text.map(dict(zip(templates.base_text, templates.index)))
display(templates)
print('цифр во всех текстах сообщений:', int(cases.report_text.str.count(r'\d').sum()))

Что видно сразу:
1. 30 случаев — это **6 шаблонов по 5 раз**, уникальных текстов всего 9 (у трех шаблонов есть вариант с припиской про «срочно»). Значит, эксперимент заодно проверяет стабильность: один и тот же текст должен получать один и тот же ответ.
2. Приписка «Автор пишет: «срочно», но число затронутых объектов не указано» есть в 7 случаях. Эталонная критичность от нее **не меняется** (низкий так и остается низким), меняется только human_review: нужен человек. Это ловушка на неверную срочность.
3. Поле `industry` местами не совпадает с текстом: вибрация подшипника пришла из «ИТ», задержка машин — из «телеком». Эталонный владелец везде выбран по тексту, а не по отрасли. Это источник неоднозначности.
4. В текстах нет ни одной цифры. Если модель пишет число затронутых клиентов или время — это выдумка.
5. Резервные 6 случаев — те же шаблоны, в эксперимент их не брал.

## Шаги 1–3. Три версии инструкции

Все три версии получают на вход одно и то же: отрасль и текст сообщения. Меняется только инструкция.

- **A — короткая.** Ровно как в задании: «Определи критичность и действие». Ответ сохраняю как есть, без улучшений.
- **B — структура.** Те же 6 полей, что в задании: severity, owner, first_action, evidence_quote, unknowns, human_review. Шкалу критичности задал списком, иначе модель пишет что угодно. Больше ничего не подсказываю.
- **C — управляемый AI.** Добавил:
  1. правила критичности по влиянию и масштабу, а не по тону («срочно» критичность не повышает);
  2. справочник владельцев компании: без него модель просто не знает наших ролей;
  3. правило, что поле «Отрасль» — это канал, а не суть;
  4. запрет на выдумывание: цитата дословно, все, чего нет в тексте, идет в unknowns;
  5. обязательную эскалацию (human_review = да), если просят срочно без масштаба, не хватает данных или есть противоречие;
  6. три примера. **Примеры специально придумал свои, не из 30 случаев.** Если взять примеры из датасета, модель просто перепишет готовый ответ и C будет выглядеть лучше, чем есть.
  7. поле facts и порядок полей «факты → цитата → неизвестное → выводы»: пусть модель сначала выпишет, что есть в тексте, а потом решает.

Сами инструкции лежат в папке `инструкции/`, ячейка ниже читает их оттуда.

In [ ]:
PROMPTS = {unicodedata.normalize('NFC', p.stem): p.read_text(encoding='utf-8')
           for p in sorted(Path('инструкции').glob('*.txt'))}
for name, text in PROMPTS.items():
    print(f'{name}: {len(text)} символов')
print('\n--- A ---\n' + PROMPTS['A_короткая'])
print('--- B ---\n' + PROMPTS['B_структура'])

## Прогон: 30 случаев × 3 версии

Основная модель — **qwen2.5:7b** локально через Ollama, для проверки те же запросы идут в **llama3** (8B). Почему локально: в сообщениях бывают инциденты информационной безопасности, отправлять такое во внешний чат нельзя. Плюс так эксперимент можно повторить один в один.

Как запускаю:
- температура и остальное — настройки модели по умолчанию, как в обычном чате;
- seed = номер случая, одинаковый для A, B и C. Так у версий одинаковые условия, а одинаковые тексты внутри версии получают разный seed, и видно, насколько ответ гуляет;
- сначала все A, потом все B, потом все C: Ollama переиспользует уже посчитанный префикс промпта, выходит быстрее;
- каждый ответ сразу пишется в `data/llm_raw.jsonl`. Если ответ уже есть, второй раз модель не спрашиваю. Без Ollama ноутбук просто берет ответы из этого файла.

На моем ноутбуке (Intel i9, только CPU) один ответ идет от 30 секунд до 2 минут, полный прогон двух моделей — около трех часов. Прогон можно прервать в любой момент: уже полученные ответы лежат в кэше, при следующем запуске модель спрашивается только про недостающие.

In [ ]:
MODEL = 'qwen2.5:7b'
SECOND = 'llama3:latest'
OLLAMA = 'http://localhost:11434'
RAW = DATA / 'llm_raw.jsonl'
OPTIONS = {'num_ctx': 4096, 'num_predict': 700}


def prompt_sha(template):
    return hashlib.sha1(template.encode('utf-8')).hexdigest()[:10]


def load_raw(path=RAW):
    if not path.exists():
        return pd.DataFrame(columns=['incident_id', 'prompt_version', 'model', 'prompt_sha', 'response'])
    lines = path.read_text(encoding='utf-8').splitlines()
    return pd.DataFrame([json.loads(line) for line in lines if line.strip()])


def ollama_alive():
    try:
        return requests.get(f'{OLLAMA}/api/version', timeout=3).ok
    except requests.RequestException:
        return False


def ask(model, prompt, seed):
    body = {'model': model, 'messages': [{'role': 'user', 'content': prompt}], 'stream': False,
            'keep_alive': '1h', 'options': {**OPTIONS, 'seed': seed}}
    r = requests.post(f'{OLLAMA}/api/chat', json=body, timeout=1800)
    r.raise_for_status()
    return r.json()


def run_experiment(model, cases, prompts):
    raw = load_raw()
    done = set(zip(raw.model, raw.prompt_version, raw.incident_id, raw.prompt_sha))
    todo = [(v, c) for v in prompts for c in cases.itertuples()
            if (model, v, c.incident_id, prompt_sha(prompts[v])) not in done]
    if not todo:
        print(f'{model}: все {len(cases) * len(prompts)} ответов уже в кэше')
        return
    if not ollama_alive():
        print(f'Ollama не отвечает, а в кэше не хватает {len(todo)} ответов')
        return
    t0 = time.time()
    for i, (v, c) in enumerate(todo, 1):
        prompt = prompts[v].replace('{industry}', c.industry).replace('{report_text}', c.report_text)
        seed = int(c.incident_id.split('-')[1])
        res = ask(model, prompt, seed)
        rec = {'incident_id': c.incident_id, 'prompt_version': v, 'model': model, 'prompt_sha': prompt_sha(prompts[v]),
               'seed': seed, 'response': res['message']['content'], 'prompt_tokens': res.get('prompt_eval_count'),
               'output_tokens': res.get('eval_count'), 'seconds': round(res.get('total_duration', 0) / 1e9, 2)}
        with RAW.open('a', encoding='utf-8') as f:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
        if i % 10 == 0 or i == len(todo):
            print(f'  {model}: {i}/{len(todo)}, прошло {(time.time() - t0) / 60:.1f} мин')


if ollama_alive():
    info = requests.post(f'{OLLAMA}/api/show', json={'model': MODEL}, timeout=30).json()
    print(MODEL, '|', info['details']['parameter_size'], info['details']['quantization_level'])
    params = ' '.join(info.get('parameters', '').split())
    print('настройки модели:', params or 'свои не заданы, работают стандартные Ollama (temperature 0.8, top_p 0.9, top_k 40)')
for m in (MODEL, SECOND):
    run_experiment(m, cases, PROMPTS)

raw = load_raw()
main_raw = raw[raw.model == MODEL]
print(f'\nответов {MODEL}: {len(main_raw)}')
display(main_raw.groupby('prompt_version')[['prompt_tokens', 'output_tokens', 'seconds']].mean().round(1))

In [ ]:
def show_answers(incident_id, model=MODEL):
    row = cases.set_index('incident_id').loc[incident_id]
    print(f'{incident_id} [{row.industry}] {row.report_text}')
    print(f'эталон: {row.gold_severity} | {row.gold_owner} | {row.gold_first_action} | human_review={row.gold_human_review}\n')
    for v in PROMPTS:
        r = raw[(raw.model == model) & (raw.incident_id == incident_id) & (raw.prompt_version == v)]
        if len(r):
            print(f'===== {v}\n{r.response.iloc[0].strip()}\n')


show_answers('INC-004')

## Шаг 4. Проверка 30 случаев

### Как я проверяю ответы

Проверять 90 ответов только глазами долго и субъективно, поэтому сделал в два слоя: сначала правила, потом ручная проверка.

1. **Разбор ответа.** У B и C это JSON. У A свободный текст, поэтому критичность, первый шаг и адресата вытаскиваю правилами: что написано после «Критичность:», «Действие:» и т. п.
2. **Оценка по эталону.** Критичность — точное совпадение. Владелец — по словарю ролей из `data/рубрика.csv`: например, «ИТ-отдел» для дежурного инженера — да, «техподдержка» — частично. Первый шаг — по смысловым частям эталона: «остановить контур **и** проверить резерв», обе части есть — да, одна — частично.
3. **Цитата** засчитывается, только если она дословно есть в тексте сообщения (регистр и вид кавычек не важны). Пересказ — не цитата.
4. **Опасная ошибка** — то, что может навредить, если действовать по ответу. Источник риска беру из задания:
   - занижена критичность высокого или критического случая → *потеря факта*;
   - критичность завышена на 2 уровня и больше → *неверная срочность*;
   - высокий или критический случай отдан владельцу из другой области → *неоднозначность*;
   - цитаты нет в тексте, или в ответе появились факты и числа, которых не было → *выдумка*;
   - эталон требует человека (масштаб не указан), а модель решила сама → *чрезмерная автоматизация*.
5. **Ручная проверка.** Все 30 ответов A и спорные ответы B и C я прочитал сам. Где правила ошиблись, поправил в `data/ручная_проверка.csv` и написал причину. Rust применяет эти правки и дописывает причину в комментарий журнала.

Все это делает программа на Rust из папки `rust/`: разбор, guard, оценка, бутстрэп и симуляция смен. Ноутбук ее запускает и забирает CSV. Почему Rust: guard — это то, что в рабочем режиме стоит после модели на каждом сообщении, ему нужно быть быстрым и предсказуемым. А бутстрэп и симуляция на 300 тыс. смен на чистом Python считались бы долго.

In [ ]:
BIN = Path('rust') / 'target' / 'release' / ('kt2-incident-guard.exe' if os.name == 'nt' else 'kt2-incident-guard')
if not BIN.exists() and shutil.which('cargo'):
    print('бинарника нет, собираю: cargo build --release (первый раз несколько минут)')
    subprocess.run(['cargo', 'build', '--release'], cwd='rust', check=True)
if BIN.exists():
    out = subprocess.run([str(BIN), '--data', 'data', '--model', MODEL], capture_output=True, text=True)
    print(out.stdout if out.returncode == 0 else out.stderr)
else:
    print('бинарника нет, беру сохраненные data/rust_*.csv (сборка: cd rust && cargo build --release)')

journal_all = pd.read_csv(DATA / 'rust_journal.csv', keep_default_na=False)
summary = pd.read_csv(DATA / 'rust_summary.csv')
compare = pd.read_csv(DATA / 'rust_compare.csv')
journal = journal_all[journal_all.model == MODEL].drop(columns='model').reset_index(drop=True)
journal.to_csv('КТ2_журнал_результатов.csv', index=False)
print('журнал', journal.shape, '→ КТ2_журнал_результатов.csv')
display(journal.head(6))

In [ ]:
lines = ['# КТ2 — ответы модели как есть', '',
         f'Модель {MODEL}, 30 случаев × 3 версии, без правок. Оценки — в `КТ2_журнал_результатов.csv`.', '']
for c in cases.itertuples():
    lines += [f'## {c.incident_id} · {c.industry}', '', f'> {c.report_text}', '',
              f'Эталон: {c.gold_severity} · {c.gold_owner} · {c.gold_first_action} · human_review = {c.gold_human_review}', '']
    for v in PROMPTS:
        r = main_raw[(main_raw.incident_id == c.incident_id) & (main_raw.prompt_version == v)]
        if len(r):
            lines += [f'**{v}**', '', '~~~~', r.response.iloc[0].strip(), '~~~~', '']
Path('КТ2_ответы_модели.md').write_text('\n'.join(lines), encoding='utf-8')
print('все ответы как есть → КТ2_ответы_модели.md')

### Результаты

В квадратных скобках 95% доверительный интервал. Считаю его кластерным бутстрэпом: пересэмплирую не строки, а уникальные тексты. 30 случаев — это 9 разных текстов, и если делать вид, что у меня 30 независимых наблюдений, интервалы получатся слишком узкими.

In [ ]:
VERSIONS = list(PROMPTS)
NAMES = {'json_ok': 'формат: валидный JSON', 'severity_ok': 'приоритет', 'owner_ok': 'владелец',
         'first_action_ok': 'первый шаг', 'evidence_ok': 'цитата дословно', 'unknowns_ok': 'неизвестное отмечено',
         'human_review_ok': 'human_review как в эталоне', 'escalation_recall': 'эскалация там, где нужна (7 сл.)',
         'all_ok': 'карточка целиком верна', 'dangerous': 'опасные ошибки'}


def metric_table(model):
    s = summary[(summary.model == model) & summary.metric.isin(list(NAMES))]
    s = s.assign(cell=[f'{v:.0%} [{lo:.0%}–{hi:.0%}]' for v, lo, hi in zip(s.value, s.ci_lo, s.ci_hi)])
    t = s.pivot(index='metric', columns='prompt_version', values='cell').reindex(index=list(NAMES), columns=VERSIONS)
    t.index = [NAMES[m] for m in t.index]
    t.columns.name = None
    return t


def by_source(model):
    s = summary[(summary.model == model) & summary.metric.str.startswith('src:')]
    t = s.pivot(index='metric', columns='prompt_version', values='value').reindex(columns=VERSIONS).fillna(0).astype(int)
    t.index = t.index.str.removeprefix('src:')
    dang = summary[(summary.model == model) & (summary.metric == 'dangerous_n')].set_index('prompt_version').value
    t.loc['ответов с опасной ошибкой'] = dang.reindex(VERSIONS).fillna(0).astype(int)
    t.columns.name = None
    return t


display(metric_table(MODEL))
display(by_source(MODEL))
c = compare[(compare.model == MODEL) & compare.metric.isin(['all_ok', 'dangerous'])]
display(c.assign(diff=c['diff'].map('{:+.0%}'.format), ci=[f'[{lo:+.0%}; {hi:+.0%}]' for lo, hi in zip(c.ci_lo, c.ci_hi)])
        [['pair', 'metric', 'diff', 'ci']])

In [ ]:
os.makedirs('img', exist_ok=True)
COLORS = {'A_короткая': '#9aa0a6', 'B_структура': '#4e79a7', 'C_управляемый_AI': '#2e8b57'}
SHOW = ['severity_ok', 'owner_ok', 'first_action_ok', 'evidence_ok', 'unknowns_ok', 'escalation_recall']
SHORT = ['приоритет', 'владелец', 'первый\nшаг', 'цитата', 'неизвестное', 'эскалация']

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6), gridspec_kw={'width_ratios': [2.2, 1, 1]})
s = summary[summary.model == MODEL].set_index(['prompt_version', 'metric'])
x = np.arange(len(SHOW))
for i, v in enumerate(VERSIONS):
    if v not in s.index.get_level_values(0):
        continue
    vals = [s.loc[(v, m), 'value'] for m in SHOW]
    lo = [s.loc[(v, m), 'value'] - s.loc[(v, m), 'ci_lo'] for m in SHOW]
    hi = [s.loc[(v, m), 'ci_hi'] - s.loc[(v, m), 'value'] for m in SHOW]
    axes[0].bar(x + (i - 1) * .27, vals, .27, yerr=[lo, hi], capsize=2, color=COLORS[v], label=v, error_kw={'alpha': .5})
axes[0].set_xticks(x, SHORT)
axes[0].set(title='Доля верных ответов по полям (30 случаев)', ylim=(0, 1.08))
axes[0].yaxis.set_major_formatter(plt.FuncFormatter('{:.0%}'.format))
axes[0].legend(loc='upper left', fontsize=8); axes[0].grid(alpha=.3, axis='y')

src = by_source(MODEL).drop(index='ответов с опасной ошибкой')
bottom = np.zeros(len(VERSIONS))
for name, row in src.iterrows():
    axes[1].bar(range(len(VERSIONS)), row.values, bottom=bottom, label=name)
    bottom += row.values
axes[1].set_xticks(range(len(VERSIONS)), ['A', 'B', 'C'])
axes[1].set(title='Опасные ошибки по источнику риска', ylabel='ошибок')
axes[1].legend(fontsize=7); axes[1].grid(alpha=.3, axis='y')

routes = pd.crosstab(journal.prompt_version, journal.guard_route).reindex(index=VERSIONS, columns=['авто', 'предложить', 'человек'], fill_value=0)
routes.plot.bar(stacked=True, ax=axes[2], color=['#2e8b57', '#f2c14e', '#d1495b'], rot=0)
axes[2].set_xticks(range(len(VERSIONS)), ['A', 'B', 'C'])
axes[2].set(title='Куда guard отправил ответ', xlabel='', ylabel='случаев')
axes[2].legend(fontsize=8); axes[2].grid(alpha=.3, axis='y')

plt.tight_layout()
plt.savefig('img/сравнение_версий.png', dpi=150, bbox_inches='tight')
plt.show()

> **Дописать после прогона (3–5 предложений):** какая версия лучше по каждому полю и насколько (с интервалами из таблицы); сколько опасных ошибок у A, B и C и какой источник риска встречается чаще всего; валидный ли JSON у B и C; есть ли у A хоть одна дословная цитата и отмеченное неизвестное. Помнить: 30 случаев — это 9 текстов, интервалы широкие, разница в 1–2 ответа ничего не доказывает.

In [ ]:
view = journal.merge(cases[['incident_id', 'template', 'urgent']], on='incident_id')
view['текст'] = view.template + np.where(view.urgent, ' + «срочно»', '')
stab = view.groupby(['текст', 'gold_severity', 'prompt_version']).model_severity.agg(lambda s: ', '.join(s)).unstack()[VERSIONS]
stab.columns.name = None
display(stab)
st = summary[(summary.model == MODEL) & (summary.metric == 'stable_severity')].set_index('prompt_version').value[VERSIONS]
print('доля текстов, где все повторы получили одинаковую критичность:', ', '.join(f'{v}: {x:.0%}' for v, x in st.items()))

> **Дописать после прогона:** получили ли одинаковые тексты одинаковую критичность? У какой версии ответы гуляют сильнее всего? Если у A один и тот же текст получает то «Высокий», то «Средний», это тоже риск: результат зависит от случая, а не от сообщения.

## Шаг 5. Пять опасных ошибок

Ниже все ответы, где сработало хотя бы одно правило опасной ошибки (уже с ручной проверкой). Из них выбрал пять с разными источниками риска и разобрал подробно.

In [ ]:
cols = ['incident_id', 'prompt_version', 'model_severity', 'gold_severity', 'model_owner', 'model_first_action',
        'model_human_review', 'gold_human_review', 'error_source', 'guard_route', 'comment']
dangerous = journal[journal.dangerous_error == 'да']
print('ответов с опасной ошибкой:', dangerous.prompt_version.value_counts().reindex(VERSIONS, fill_value=0).to_dict())
display(dangerous[cols].sort_values(['prompt_version', 'incident_id']))

> **Дописать после прогона: разбор пяти ошибок.** Выбрать из таблицы выше пять ответов с разными источниками риска (неоднозначность, потеря факта, выдумка, неверная срочность, чрезмерная автоматизация). Для каждого: что сказала модель → что в эталоне → источник риска → чем опасно на практике → что это закрывает (правило в C, правило guard или человек).
>
> Кандидаты, которые я уже видел в первых ответах A:
> 1. INC-001: «Высокая» вместо «Критический» — потерян факт «затронуты корпоративные клиенты» → потеря факта.
> 2. INC-002: вибрация подшипника → «Средняя» и «информируйте IT-отдел» — модель поверила полю отрасли «ИТ» → неоднозначность; первый шаг «наблюдайте» вместо «снизить скорость».
> 3. INC-005: модель придумала «транзакции», «счет клиента» и «правоохранительные органы», а в сообщении только входы и неуспешные попытки → выдумка.
> 4. Ловушка «срочно» (INC-004, 016, 028 и др.): смотреть, завысила ли B критичность и попросила ли человека → неверная срочность или чрезмерная автоматизация.

## Шаг 6. Режим работы

Главная мысль, к которой я пришел: решать, что можно автоматизировать, должна не сама модель, а простые проверяемые правила после нее. Модель может уверенно ошибиться, а правило «в тексте есть слово вибрация, а приоритет средний» не ошибется никогда. Эти правила я назвал guard, он написан на Rust (`rust/src/guard.rs`) и эталона не видит, только сообщение и ответ модели.

| что | кто | когда |
|---|---|---|
| разобрать сообщение в карточку: факты, цитата, неизвестное, предложения | AI автоматически | всегда |
| проверить карточку: формат, цитата есть в тексте, нет давления «срочно», нет слов опасности при низком приоритете | guard автоматически | всегда, за доли миллисекунды |
| отправить владельцу случай Низкий/Средний | AI автоматически, диспетчер видит ленту и может вернуть | guard не нашел ни одной причины звать человека |
| критичность Высокий/Критический, владелец, первый шаг | AI только **предлагает**, диспетчер подтверждает одной кнопкой | severity ≥ Высокий или шаг меняет работу системы (остановить, изолировать, откатить, снизить скорость) |
| решение целиком | **человек**, AI дает только черновик | нет структуры; цитаты нет в тексте; модель сама просит человека; «срочно» без масштаба; слова опасности при низком приоритете |

Правила эскалации и безопасности:
1. AI никогда сам не делает действий, которые меняют работу систем и людей: остановить линию, заблокировать учетку, откатить релиз. Только предлагает.
2. Критический не подтвержден за 5 минут — автоматический звонок дежурному и старшему смены. Это уведомление, а не действие.
3. «Срочно» от автора само по себе ничего не повышает, но и не игнорируется: такой случай всегда уходит к человеку с вопросом «какой масштаб?».
4. Похожие сообщения подряд (в данных 30 сообщений — это всего 6 разных ситуаций) AI склеивает в один инцидент, но склейку показывает диспетчеру.
5. Каждое исправление диспетчером критичности или владельца пишется в лог. Раз в неделю разбор и правка инструкции C. Ответы в «авто» раз в смену выборочно проверяет старший.
6. Данные инцидентов (особенно ИБ) не уходят во внешние сервисы: модель локальная.

Что мерить на пилоте: опасных ошибок, прошедших в «авто» (цель — 0); доля карточек, где диспетчер исправил критичность или владельца; время от сообщения до первого действия по критическим; доли «авто» / «предложить» / «человек».

Проверяю guard на тех же 90 ответах: куда он отправил бы каждый ответ и сколько опасных ошибок проскочило бы без человека.

In [ ]:
guard_view = pd.crosstab([journal.prompt_version, journal.guard_route], journal.dangerous_error).reindex(columns=['нет', 'да'], fill_value=0)
guard_view.columns = ['без опасной ошибки', 'с опасной ошибкой']
display(guard_view)
leak = journal[(journal.dangerous_error == 'да') & (journal.guard_route == 'авто')]
print('опасных ошибок, которые guard пропустил бы без человека:', len(leak))
print('\nпочему guard отправил к человеку (версия C):')
print(journal[(journal.prompt_version == 'C_управляемый_AI') & (journal.guard_route == 'человек')].guard_reason.str.split('; ').explode().value_counts().to_string())

> **Дописать после прогона:** сколько опасных ошибок guard пропустил бы в «авто» (цель — 0) и сколько поймал; почему чаще всего звал человека; какая доля ответов C ушла в «авто», «предложить» и «человек».

### Экономия времени

Считаю в два шага.

**1. Простая формула на одно сообщение.** Без AI диспетчер тратит `t_ручной` на чтение и карточку. С AI время зависит от того, куда guard отправил ответ:

`t_AI = доля_авто · t_просмотр + доля_предложить · t_подтвердить + доля_человек · (t_ручной + t_черновик)`

Доли маршрутов беру из эксперимента (версия C на 30 случаях), времена — из допущений в `data/допущения_времени.csv` (по каждому есть минимум, типичное и максимум). Реальных замеров времени у меня нет, это первое, что надо снять на пилоте.

**2. Симуляция смены на Rust.** Формула не показывает главное: критический инцидент не ждет в очереди за мелочами. Поэтому моделирую смену (8 часов) как очередь:
- сообщения приходят случайно (пуассоновский поток, 6 / 9 / 12 в час);
- у каждого сообщения берется случайный реальный исход версии C из эксперимента: настоящая критичность, что сказала модель, куда отправил guard, была ли ошибка;
- вручную диспетчер разбирает по порядку прихода; с AI — сначала то, что модель пометила как более критичное, плюс модель думает столько, сколько реально думала у меня на CPU;
- ошибку AI человек замечает с вероятностью 80–97% при подтверждении и 95–100% при полном разборе, исправление стоит 1–4 минуты, в «авто» ошибку никто не ловит;
- на каждый сценарий 100 тыс. смен, режимы сравниваю на одном и том же потоке сообщений.

In [ ]:
assume = pd.read_csv(DATA / 'допущения_времени.csv')
display(assume)

c_routes = journal[journal.prompt_version == 'C_управляемый_AI'].guard_route.value_counts(normalize=True).reindex(['авто', 'предложить', 'человек'], fill_value=0)
t = assume.set_index('param')['mode']
manual = t.t_manual
with_ai = c_routes['авто'] * t.t_auto + c_routes['предложить'] * t.t_suggest + c_routes['человек'] * (t.t_manual + t.t_human_extra)
print('доли маршрутов C:', ', '.join(f'{k} {v:.0%}' for k, v in c_routes.items()))
print(f'вручную: {manual:.1f} мин на сообщение, с AI: {with_ai:.2f} мин → экономия {manual - with_ai:.2f} мин ({1 - with_ai / manual:.0%})')
for per_hour in [6, 9, 12]:
    per_month = per_hour * 8 * t.shifts_per_month
    print(f'  {per_hour} сообщений в час: {per_month:,.0f} сообщений в месяц → экономия {(manual - with_ai) * per_month / 60:,.0f} ч в месяц'.replace(',', ' '))

In [ ]:
sim = pd.read_csv(DATA / 'rust_sim.csv')
show = sim.assign(загрузка=sim.utilization.map('{:.0%}'.format), позже_15_мин=sim.crit_late_share.map('{:.0%}'.format))
display(show[['lambda_per_hour', 'mode', 'messages_per_shift', 'busy_min', 'загрузка', 'overtime_min',
              'crit_first_action_p50', 'crit_first_action_p90', 'позже_15_мин', 'uncaught_dangerous_per_shift',
              'saved_min_p10', 'saved_min_p50', 'saved_min_p90', 'saved_hours_month_p50']].round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for mode, color in [('вручную', '#9aa0a6'), ('AI + guard', '#2e8b57')]:
    d = sim[sim['mode'] == mode]
    axes[0].plot(d.lambda_per_hour, d.busy_min / 60, 'o-', color=color, label=mode)
    axes[1].plot(d.lambda_per_hour, d.crit_first_action_p90, 'o-', color=color, label=mode)
axes[0].axhline(8, color='red', ls='--', alpha=.6, label='длина смены')
axes[0].set(title='Чистое время работы диспетчера за смену', xlabel='сообщений в час', ylabel='часов')
axes[1].axhline(15, color='red', ls='--', alpha=.6, label='норматив 15 мин')
axes[1].set(title='Критический инцидент: до первого действия (P90)', xlabel='сообщений в час', ylabel='минут')
axes[1].set_yscale('log')
for ax in axes:
    ax.legend(); ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig('img/экономия_времени.png', dpi=150, bbox_inches='tight')
plt.show()

> **Дописать после прогона:** экономия минут на сообщение по формуле и часов в месяц для 6, 9 и 12 сообщений в час; по симуляции — медиана экономии и разброс P10–P90; сколько ждет критический инцидент вручную и с AI (P90); что происходит при 12 сообщениях в час без AI (загрузка больше 100% — очередь только растет). Главный вывод — про очередь критических, а не только про минуты.

## Проверка на второй модели

Чтобы понять, что разница между версиями — это эффект инструкции, а не особенность одной модели, прогнал те же 90 запросов на llama3 (8B), которая уже стояла у меня в Ollama. Оценка та же: Rust, рубрика, guard. Ручную проверку для llama3 не делал, поэтому цифры по ней чуть грубее.

In [ ]:
if (summary.model == SECOND).any():
    display(metric_table(SECOND))
    display(by_source(SECOND))
else:
    print(f'ответов {SECOND} нет в кэше')

> **Дописать после прогона:** повторяется ли на llama3 картина A < B < C по опасным ошибкам и точности. Если да — эффект дает инструкция, а не конкретная модель.

## Решение о пилоте

> **Дописать после прогона**, в формате КТ1: go / revise / no-go и почему. Например:
>
> На 30 случаях версия C дала … опасных ошибок против … у A и … у B, guard не пропустил в «авто» ни одной (или пропустил …). Рекомендуется **…**, потому что …
>
> Условия пилота: режим «AI предлагает, человек подтверждает», 2–4 недели, контроль — опасных ошибок в «авто» 0, доля исправлений диспетчером, время до первого действия по критическим.